# SignSight — Train CAR35 — nhãn đã rà R3 với YOLO11s

Dùng ZIP **CAR35 V4 đã chia sẵn** và ZIP patch R3; không cần SupSign gốc, không chia lại train/val/test.
Mặc định tạo revision đã rà nhãn: 36.980 ảnh / 40.522 bbox. Không thêm 461 ảnh ViTSD chưa rà.
Tải dataset bằng **gdown**; không mount Drive. Notebook chạy độc lập, không clone repo.
Chọn GPU runtime. Dữ liệu chỉ nằm trong `/content`; lưu notebook không lưu dataset hay checkpoint.
Split: 25.886 train / 5.547 val / 5.547 test; 35 lớp. Có ảnh cùng cảnh giữa split và lớp hiếm ít mẫu holdout.
Đây là split phát triển, chưa phải test thực địa độc lập. Không so trực tiếp mAP với V3.


In [ ]:
import sys, subprocess, os
os.environ['YOLO_AUTOINSTALL'] = 'false'
os.environ['WANDB_MODE'] = 'disabled'
packages = ['Pillow>=10,<13', 'matplotlib>=3.8,<4', 'PyYAML>=6,<7', 'gdown==5.2.0']
packages.append("ultralytics==8.4.173")
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])


## 1. Cấu hình

Upload ZIP CAR35 đã được cung cấp lên Drive, chia sẻ file cho phép tải bằng link, rồi dán link vào SOURCE_URL.
Giữ nguyên checksum đã ghim. Runtime/tài khoản mới chỉ cần notebook và link ZIP; gdown sẽ tải lại.
ALLOW_REPACKED_SOURCE=True cho phép ZIP V4 nén lại khi manifest khớp chính xác bản đã ghim.
Toàn bộ ảnh/nhãn vẫn phải vượt qua kiểm tra hash, inventory, lớp và split trước khi train.
Nếu manifest khác, notebook dừng và in hash thực tế; không tự nhận một dataset lạ.
APPLY_REVIEWED_PATCH=True mặc định: thêm PATCH_URL là link ZIP patch nhỏ. False dùng baseline V4.
Checksum patch và manifest revision được điền sẵn; không đổi chúng cho một file khác.
V4 được giải nén một lần vào staging rồi sửa nhãn trên bản mới; không đổi ảnh, nhóm hay split.
Runtime mới cần tải V4 và patch lại. Giữ bản V4 để so sánh; không resume run V4 với nhãn revision.
Không có DOWNLOAD=False để bỏ qua dataset ở runtime mới. Runtime cũ có dataset đã xác minh thì dùng lại.
Kết quả không tự lưu lên Drive. Tải ZIP kết quả trước khi runtime bị xóa.


In [ ]:
from pathlib import Path
import sys, os
WORK = Path('/content/signsight_car35')
WORK.mkdir(parents=True, exist_ok=True)
(WORK / '.ultralytics').mkdir(exist_ok=True)
os.environ['YOLO_CONFIG_DIR'] = str(WORK / '.ultralytics')
SOURCE_URL = ''  # Dán link file ZIP CAR35 trên Drive, không phải folder
DATASET_SHA256 = '78ff592aaf49d9539a92fd02334f6394a22ac1a592c32008a1416e4fe481e872'
EXPECTED_MANIFEST_SHA256 = '7d89760a0e83628a40895ee6a6367315aebb79b908a0101f50b66801479b1901'
ALLOW_REPACKED_SOURCE = True  # Chỉ chấp nhận nén lại nếu manifest/ảnh/nhãn vẫn đúng bản đã ghim
APPLY_REVIEWED_PATCH = True  # False chỉ khi cần chạy baseline V4 gốc
PATCH_URL = ''  # Link ZIP patch R3, không phải dataset ZIP
PATCH_SHA256 = '9f0a96250b345cf0d324d8b23c628e203260551d801c39b25555c4bcd1e2632a'
REVISED_DATASET_VERSION = 'car35_v4_reviewed_r3_2026-11-19'
EXPECTED_REVISED_MANIFEST_SHA256 = 'a62a286e5ce45a508d158423ec3efa559a857d385326f19c53280fc667f4f66c'
EXPORT_REVISED_DATASET = False  # Chỉ dùng trong EDA; True sẽ tạo thêm ZIP dataset lớn
DATASET_ROOT = WORK / 'dataset'
RUN_NAME = 'yolo11s_car35_reviewed_r3_640' if APPLY_REVIEWED_PATCH else 'yolo11s_car35_v4_640'
RUN_DIR = WORK / 'runs' / RUN_NAME
EPOCHS = 80
IMAGE_SIZE = 640
BATCH = 8
DEVICE = '0'
RESUME = False  # Chỉ True khi RUN_DIR còn last.pt và training_request.json
RUN_HELDOUT_TEST = False  # Chỉ bật sau khi chốt cấu hình bằng validation
DOWNLOAD_RESULTS = True
assert RUN_NAME and Path(RUN_NAME).name == RUN_NAME and RUN_NAME not in ('.', '..')
print('Dataset local:', DATASET_ROOT)


In [ ]:
# Helpers được nhúng để notebook chạy độc lập sau khi upload.
import hashlib, importlib, json, shutil
TRAINING_FILES = {'training/__init__.py': '"""Offline datasets, experiments and notebook tooling; never imported by the API."""\n', 'training/data/__init__.py': '', 'training/experiments/__init__.py': '', 'training/data/common.py': "import hashlib, json\nfrom pathlib import Path\n\nCAR35 = [\n    'Speed_limit_40', 'Speed_limit_50', 'Speed_limit_60', 'Speed_limit_70', 'Speed_limit_80',\n    'No_entry', 'No_cars', 'No_cars_turn_left', 'No_cars_turn_right',\n    'No_turn_left', 'No_turn_right', 'No_U_turn', 'No_cars_U_turn',\n    'No_straight_ahead', 'No_turn_left_and_U_turn', 'No_turn_left_and_right',\n    'No_parking', 'No_stopping_and_parking', 'No_parking_odd_days', 'No_parking_even_days',\n    'Go_straight', 'Keep_right', 'Roundabout', 'One_way_road', 'Road_reserved_for_cars',\n    'Pedestrian_crossing', 'Children_crossing', 'Road_works', 'Slow_down',\n    'Dangerous_curve_left', 'Dangerous_curve_right',\n    'No_overtaking', 'End_no_overtaking', 'Give_way', 'End_all_prohibitions',\n]\nSPLITS = ('train', 'val', 'test')\nVERSION = 2\n\n\ndef digest_file(path):\n    with Path(path).open('rb') as stream:\n        digest = hashlib.sha256()\n        for block in iter(lambda: stream.read(1024*1024), b''): digest.update(block)\n        return digest.hexdigest()\n\n\ndef write_json(path, value):\n    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding='utf-8'))\n\n\ndef recipe(classes=CAR35, seed=42, block_size=100, near_threshold=4, groups=None):\n    from training.compatibility import implementation_digest\n    group_digest = hashlib.sha256(json.dumps(groups, sort_keys=True).encode()).hexdigest()\n    return dict(version=VERSION, classes=list(classes), seed=seed, block_size=block_size,\n        near_threshold=near_threshold, groups_sha256=group_digest,\n        implementation_sha256=implementation_digest(),\n        reader_sha256=hashlib.sha256(Path(__file__).with_name('audit.py').read_text(encoding='utf-8').encode()).hexdigest())\n", 'training/data/downloads.py': '"""Download a shared Drive ZIP to local Colab storage, without account cookies."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport re\nfrom urllib.parse import parse_qs, urlparse\nimport zipfile\n\n\nclass ZipChecksumMismatch(ValueError):\n    """Complete ZIP retained for diagnosis, never published as the expected file."""\n\n\ndef drive_file_id(source):\n    source = str(source).strip()\n    if not source or \'YOUR_\' in source or \'PASTE_\' in source:\n        raise ValueError(\'Paste a shared Google Drive file URL or file ID\')\n    if \'://\' in source:\n        parsed = urlparse(source)\n        if parsed.scheme != \'https\' or parsed.hostname != \'drive.google.com\' or parsed.username:\n            raise ValueError(\'Use a Google Drive file link, not a folder or another website\')\n        match = re.fullmatch(r\'/file/d/([A-Za-z0-9_-]+)(?:/.*)?\', parsed.path)\n        if match:\n            source = match[1]\n        elif parsed.path in (\'/uc\', \'/open\'):\n            source = parse_qs(parsed.query).get(\'id\', [\'\'])[0]\n        else:\n            raise ValueError(\'Use a Google Drive file link, not a folder\')\n    if not re.fullmatch(r\'[A-Za-z0-9_-]{10,}\', source):\n        raise ValueError(\'Invalid Google Drive file ID\')\n    return source\n\n\ndef download_drive_zip(source, target, *, expected_sha256=\'\', downloader=None):\n    """Publish only a complete ZIP; source receipts prevent resuming another file.\n\n    gdown owns its resumable temporary files. This helper keeps those on network\n    failure and never overwrites a pre-existing target from an unknown source.\n    An authoritative SHA-256 can adopt an already downloaded matching ZIP.\n    """\n    file_id = drive_file_id(source)\n    checksum = str(expected_sha256).strip().lower()\n    if checksum and not re.fullmatch(\'[a-f0-9]{64}\', checksum):\n        raise ValueError(\'Expected SHA-256 must have 64 hexadecimal characters\')\n    target = Path(target).resolve()\n    receipt = target.with_name(target.name+\'.source.json\')\n    staging = target.with_name(target.name+\'.download\')\n\n    def validate(path):\n        if not path.is_file() or not zipfile.is_zipfile(path):\n            raise ValueError(\'Download is not a ZIP. Check Drive sharing, quota and the file link.\')\n        if checksum:\n            with path.open(\'rb\') as stream:\n                actual = hashlib.file_digest(stream, \'sha256\').hexdigest()\n            if actual != checksum:\n                raise ZipChecksumMismatch(f\'Downloaded ZIP SHA-256 mismatch for {path.name}: \'\n                    f\'expected {checksum}; actual {actual}. Complete ZIP kept at {path}; check the source link/version.\')\n\n    known = receipt.exists()\n    if known:\n        try:\n            same_source = json.loads(receipt.read_text())[\'file_id\'] == file_id\n        except (ValueError, KeyError, TypeError):\n            same_source = False\n        if not same_source: raise FileExistsError(\'Local download belongs to another Drive file; choose a new path\')\n    elif target.exists():\n        if not checksum: raise FileExistsError(\'Existing ZIP has no source receipt; supply its SHA-256 or choose a new path\')\n        validate(target)\n    elif staging.exists() or any(target.parent.glob(staging.name+\'*.tmp\')):\n        raise FileExistsError(\'Unknown partial download exists; choose a new local path\')\n    target.parent.mkdir(parents=True, exist_ok=True)\n    if not known:\n        with receipt.open(\'x\', encoding=\'utf-8\') as stream:\n            json.dump({\'file_id\': file_id}, stream)\n    if target.exists():\n        validate(target)\n        return target\n    if staging.exists():\n        try:\n            validate(staging)\n        except ZipChecksumMismatch:\n            raise\n        except ValueError:\n            staging.unlink()  # Only the staging file identified by our matching receipt.\n    if not staging.exists():\n        if downloader is None:\n            import gdown\n            downloader = gdown.download\n        try:\n            result = downloader(id=file_id, output=str(staging), quiet=False,\n                                resume=True, use_cookies=False)\n        except Exception as error:\n            raise RuntimeError(\'Drive download failed. Check public-link access/quota, then rerun this cell \'\n                               \'to resume in the same runtime.\') from error\n        if result is None:\n            raise RuntimeError(\'Drive returned no download. Check public-link access and download quota.\')\n    try:\n        validate(staging)\n    except ZipChecksumMismatch:\n        raise\n    except ValueError:\n        staging.unlink(missing_ok=True)\n        raise\n    if target.exists(): raise FileExistsError(\'Download target appeared; it was not overwritten\')\n    staging.rename(target)\n    return target\n', 'training/data/car35_bundle.py': '"""Restore/verify CAR35 V4 or a reviewed revision, without raw data or re-splitting."""\nfrom collections import Counter\nfrom concurrent.futures import ThreadPoolExecutor\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path, PurePosixPath\nimport re\nimport shutil\nimport tempfile\nimport zipfile\n\nimport yaml\nfrom training.data.common import CAR35, SPLITS, digest_file\n\n\ndef inspect_car35_archive(archive, *, expected_sha256, expected_manifest_sha256, allow_repacked=False):\n    """Preflight container identity only; restore still verifies every image/label.\n\n    A changed ZIP checksum can mean compression/timestamps changed. Accept that\n    only with explicit opt-in and the exact independently pinned manifest bytes.\n    Never learn a new trusted manifest from the downloaded archive itself.\n    """\n    archive = Path(archive)\n    for label, value in ((\'archive\', expected_sha256), (\'manifest\', expected_manifest_sha256)):\n        if not isinstance(value, str) or not re.fullmatch(\'[a-fA-F0-9]{64}\', value):\n            raise ValueError(f\'CAR35 {label} SHA-256 must be pinned (64 hexadecimal characters)\')\n    expected_sha256, expected_manifest_sha256 = expected_sha256.lower(), expected_manifest_sha256.lower()\n    actual = digest_file(archive)\n    repacked = actual != expected_sha256\n    if repacked and not allow_repacked:\n        raise ValueError(f\'CAR35 ZIP SHA-256 mismatch for {archive.name}: expected {expected_sha256}; actual {actual}\')\n    with zipfile.ZipFile(archive) as source:\n        entries = [m for m in source.infolist() if m.filename.casefold() == \'manifest.json\']\n        if len(entries) != 1 or entries[0].filename != \'manifest.json\' or entries[0].file_size > 64*1024**2:\n            raise ValueError(\'Expected a flat prepared CAR35 ZIP with one manifest.json at its root; check SOURCE_URL\')\n        manifest_sha = hashlib.sha256(source.read(entries[0])).hexdigest()\n    if manifest_sha != expected_manifest_sha256:\n        raise ValueError(f\'CAR35 manifest SHA-256 mismatch for {archive.name}: expected {expected_manifest_sha256}; \'\n                         f\'actual {manifest_sha}. Check SOURCE_URL: use the prepared V4 ZIP, not raw SupSign, \'\n                         \'a patch ZIP or a training-results ZIP.\')\n    return dict(archive_sha256=actual, expected_archive_sha256=expected_sha256,\n                manifest_sha256=manifest_sha, repacked=repacked)\n\n\ndef verify_car35(root, *, expected_manifest_sha256=\'\', workers=8):\n    root=Path(root).resolve();manifest_file=root/\'manifest.json\'\n    if expected_manifest_sha256 and digest_file(manifest_file)!=expected_manifest_sha256:\n        raise ValueError(\'CAR35 manifest SHA-256 mismatch\')\n    manifest=json.loads(manifest_file.read_text(encoding=\'utf-8\'))\n    if manifest.get(\'kind\') not in (\'car35_folder_development_v4\',\'car35_reviewed_development_v1\') or manifest.get(\'names\')!=CAR35:\n        raise ValueError(\'Expected prepared CAR35 V4 taxonomy, not raw SupSign or a V3 cache\')\n    if manifest[\'kind\']==\'car35_reviewed_development_v1\':\n        from training.data.car35_revision import validate_revision_provenance\n        validate_revision_provenance(root,manifest)\n    config=yaml.safe_load((root/\'data.yaml\').read_text(encoding=\'utf-8\'))\n    if config!=dict(train=\'images/train\',val=\'images/val\',test=\'images/test\',names=CAR35):\n        raise ValueError(\'Unsafe or changed CAR35 data.yaml\')\n    if manifest.get(\'independent_field_test\') is not False:\n        raise ValueError(\'This development bundle is not an independent field test\')\n    if type(workers) is not int or not 1<=workers<=32:raise ValueError(\'Invalid worker count\')\n    images,labels,hashes,groups=set(),set(),set(),{}\n    counts={s:Counter() for s in SPLITS};sizes=Counter()\n    def check(row):\n        split=row[\'split\'];image,label=(root/row[k] for k in (\'image\',\'label\'));sha=row[\'sha256\']\n        if (split not in SPLITS or not re.fullmatch(\'[0-9a-f]{64}\',sha)\n            or image.parent!=root/\'images\'/split or image.stem!=sha\n            or image.suffix.lower() not in {\'.png\',\'.jpg\',\'.jpeg\',\'.webp\',\'.bmp\',\'.tif\',\'.tiff\'}\n            or label!=root/\'labels\'/split/(sha+\'.txt\')\n            or not image.resolve().is_relative_to(root) or not label.resolve().is_relative_to(root)):\n            raise ValueError(\'Unsafe manifest image/label path\')\n        content=label.read_bytes()\n        if digest_file(image)!=sha or hashlib.sha256(content).hexdigest()!=row[\'label_sha256\']:\n            raise ValueError(\'CAR35 image/label hash changed\')\n        current=Counter()\n        for line in content.decode(\'utf-8\').splitlines():\n            if not line.strip():continue\n            fields=line.split()\n            if len(fields)!=5:raise ValueError(\'Invalid detection label\')\n            c=int(fields[0]);x,y,w,h=map(float,fields[1:])\n            if (not 0<=c<35 or not all(math.isfinite(v) for v in (x,y,w,h)) or min(w,h)<=0\n                or min(x-w/2,y-h/2)<-1e-6 or max(x+w/2,y+h/2)>1+1e-6):\n                raise ValueError(\'Invalid CAR35 class/bbox\')\n            current[c]+=1\n        return row,image,label,current\n    with ThreadPoolExecutor(max_workers=workers) as pool:\n        for row,image,label,current in pool.map(check,manifest[\'images\']):\n            if row[\'sha256\'] in hashes or (row[\'group\'] in groups and groups[row[\'group\']]!=row[\'split\']):\n                raise ValueError(\'Duplicate hash or cross-split CAR35 group\')\n            images.add(image);labels.add(label);hashes.add(row[\'sha256\']);groups[row[\'group\']]=row[\'split\']\n            sizes[row[\'split\']]+=1;counts[row[\'split\']].update(current)\n    actual_labels=set((root/\'labels\').rglob(\'*.*\'))\n    allowed_caches={root/\'labels\'/(s+suffix) for s in SPLITS for suffix in (\'.cache\',\'.cache.npy\')}\n    for cache in actual_labels & allowed_caches:\n        if cache.is_symlink() or not cache.is_file() or not cache.resolve().is_relative_to(root):\n            raise ValueError(\'Unsafe YOLO label cache path\')\n    if images!=set((root/\'images\').rglob(\'*.*\')) or labels!=actual_labels-allowed_caches:\n        raise ValueError(\'CAR35 image/label inventory changed\')\n    if set(sizes)!=set(SPLITS):raise ValueError(\'Empty CAR35 split\')\n    for split in SPLITS:\n        recorded={int(k):v for k,v in manifest[\'class_counts\'][split].items() if v}\n        if recorded!=dict(counts[split]):raise ValueError(\'CAR35 class counts changed\')\n    if any(not counts[\'train\'][c] for c in range(35)):raise ValueError(\'Missing CAR35 train class\')\n    return dict(kind=manifest[\'kind\'],dataset_version=manifest.get(\'dataset_version\',\'car35_v4\'),\n                images=len(images),boxes=sum(sum(c.values()) for c in counts.values()),names=list(CAR35),\n                split_images=dict(sizes),split_groups=dict(Counter(groups.values())),\n                class_counts={s:dict(counts[s]) for s in SPLITS},groups=len(groups),\n                manifest_sha256=digest_file(manifest_file),independent_field_test=False,\n                limitations=manifest.get(\'limitations\',\'Development split; scene overlap may remain.\'))\n\n\ndef restore_car35_bundle(archive,output,*,expected_sha256,expected_manifest_sha256=\'\'):\n    archive,output=Path(archive).resolve(),Path(output).resolve()\n    if output.exists():raise FileExistsError(output)\n    if not re.fullmatch(\'[0-9a-fA-F]{64}\',expected_sha256 or \'\'):\n        raise ValueError(\'CAR35 archive SHA-256 is required\')\n    if digest_file(archive)!=expected_sha256.lower():raise ValueError(\'CAR35 ZIP checksum mismatch\')\n    with zipfile.ZipFile(archive) as source:\n        entries=source.infolist();seen=set()\n        for member in entries:\n            name=member.filename;path=PurePosixPath(name)\n            if (not name or path.is_absolute() or \'..\' in path.parts or \'\\\\\' in name or \':\' in name\n                or name.casefold() in seen or (member.external_attr>>16)&0o170000==0o120000\n                or member.flag_bits&1):raise ValueError(\'Unsafe or duplicate CAR35 ZIP member\')\n            seen.add(name.casefold())\n        if not {\'data.yaml\',\'manifest.json\'}<=seen:raise ValueError(\'ZIP is not a flat prepared CAR35 bundle\')\n        required=sum(m.file_size for m in entries)+1024**3\n        output.parent.mkdir(parents=True,exist_ok=True)\n        if shutil.disk_usage(output.parent).free<required:raise ValueError(\'Not enough disk space to restore CAR35\')\n        with tempfile.TemporaryDirectory(prefix=\'.car35-restore-\',dir=output.parent) as temporary:\n            stage=Path(temporary)/\'dataset\';stage.mkdir()\n            for member in entries:\n                if member.is_dir():continue\n                target=stage/member.filename;target.parent.mkdir(parents=True,exist_ok=True)\n                with source.open(member) as stream,target.open(\'wb\') as dest:shutil.copyfileobj(stream,dest)\n            info=verify_car35(stage,expected_manifest_sha256=expected_manifest_sha256)\n            stage.rename(output)\n    return info\n', 'training/data/label_patch.py': '"""Export small, reviewed CAR35 label overlays; never mutate the source dataset."""\nfrom collections import defaultdict\nimport csv\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path, PurePosixPath\nimport re\nimport tempfile\n\nfrom training.data.common import CAR35, digest_file, write_json\n\n\ndef create_label_patch(root, output, edits, *, expected_manifest_sha256):\n    """Hash-bound label replacements, without images or a standalone training YAML.\n\n    line_index refers to a physical, zero-based source label line. Replace its\n    class token, or remove a reviewed duplicate with an unchanged retained box.\n    Bounding-box text, whitespace and line endings of other labels remain intact.\n    An applying tool must create a new dataset identity and verify\n    the source hashes; copying these labels into V4 in place is not supported.\n    """\n    root, output = Path(root).resolve(), Path(output).resolve()\n    if output.is_relative_to(root) or root.is_relative_to(output):\n        raise ValueError(\'Patch output must be outside the source dataset\')\n    if output.exists():\n        raise FileExistsError(output)\n    manifest_file = root/\'manifest.json\'\n    if (not re.fullmatch(\'[a-f0-9]{64}\', expected_manifest_sha256 or \'\') or\n            digest_file(manifest_file) != expected_manifest_sha256):\n        raise ValueError(\'Source manifest changed or checksum is invalid\')\n    manifest = json.loads(manifest_file.read_text(encoding=\'utf-8\'))\n    if manifest.get(\'kind\') != \'car35_folder_development_v4\' or manifest.get(\'names\') != CAR35:\n        raise ValueError(\'Expected the original CAR35 V4 manifest and class order\')\n    rows = {r[\'sha256\']: r for r in manifest[\'images\']}\n    if len(rows) != len(manifest[\'images\']):\n        raise ValueError(\'Duplicate source image identity\')\n    if not edits:\n        raise ValueError(\'Supply at least one reviewed edit\')\n    grouped, seen = defaultdict(list), set()\n    for edit in edits:\n        sha, index = edit.get(\'image_sha256\', \'\'), edit.get(\'line_index\')\n        before, after = edit.get(\'from_class\'), edit.get(\'to_class\')\n        action = edit.get(\'action\', \'replace_class\')\n        if (not re.fullmatch(\'[a-f0-9]{64}\', sha) or sha not in rows or\n                type(index) is not int or index < 0 or edit.get(\'reviewed\') is not True or\n                any(type(c) is not int or not 0 <= c < 35 for c in (before, after)) or\n                action not in (\'replace_class\', \'remove_duplicate\') or\n                action == \'replace_class\' and before == after or\n                not isinstance(edit.get(\'reason\'), str) or not edit[\'reason\'].strip()):\n            raise ValueError(\'Invalid, unreviewed or unsupported CAR35 edit\')\n        if (sha, index) in seen:\n            raise ValueError(\'Duplicate edit for one label line\')\n        seen.add((sha, index)); grouped[sha].append(dict(edit, action=action))\n    changed, content_by_path = [], {}\n    for sha, changes in sorted(grouped.items()):\n        row = rows[sha]; split = row[\'split\']\n        if split not in (\'train\', \'val\', \'test\'):\n            raise ValueError(\'Invalid source split\')\n        image, label = root/row[\'image\'], root/row[\'label\']\n        if (image.parent != root/\'images\'/split or image.stem != sha or\n                row[\'label\'] != f\'labels/{split}/{sha}.txt\' or\n                not image.resolve().is_relative_to(root) or not label.resolve().is_relative_to(root)):\n            raise ValueError(\'Unsafe source image/label path\')\n        raw = label.read_bytes()\n        if hashlib.sha256(raw).hexdigest() != row[\'label_sha256\']:\n            raise ValueError(\'Source label changed\')\n        if digest_file(image) != sha:\n            raise ValueError(\'Source image changed\')\n        lines = raw.decode(\'utf-8\').splitlines(keepends=True)\n        original_lines, removed = lines.copy(), set()\n        for edit in changes:\n            index = edit[\'line_index\']\n            if index >= len(lines):\n                raise ValueError(\'Label line no longer exists\')\n            fields = original_lines[index].split()\n            if len(fields) != 5 or fields[0] != str(edit[\'from_class\']):\n                raise ValueError(\'Source class/label line differs from reviewed edit\')\n            x, y, w, h = map(float, fields[1:])\n            if (not all(math.isfinite(v) for v in (x, y, w, h)) or min(w, h) <= 0 or\n                    min(x-w/2, y-h/2) < -1e-6 or max(x+w/2, y+h/2) > 1+1e-6):\n                raise ValueError(\'Source bbox requires separate review\')\n            if edit[\'action\'] == \'remove_duplicate\':\n                retained = edit.get(\'retained_line_index\')\n                if (type(retained) is not int or not 0 <= retained < len(original_lines) or\n                        retained == index or (sha, retained) in seen):\n                    raise ValueError(\'Duplicate removal requires an unchanged retained label line\')\n                other = original_lines[retained].split()\n                if (len(other) != 5 or other[0] != str(edit[\'to_class\']) or\n                        tuple(map(float, other[1:])) != (x, y, w, h)):\n                    raise ValueError(\'Retained duplicate must have the reviewed class and identical bbox\')\n                removed.add(index)\n            else:\n                lines[index] = re.sub(r\'^(\\s*)\\d+\', lambda m: m[1]+str(edit[\'to_class\']), lines[index], count=1)\n        updated = \'\'.join(line for i, line in enumerate(lines) if i not in removed).encode(\'utf-8\')\n        content_by_path[row[\'label\']] = updated\n        changed.append(dict(image_sha256=sha, split=split, label=row[\'label\'],\n                            old_sha256=row[\'label_sha256\'], new_sha256=hashlib.sha256(updated).hexdigest(),\n                            edits=changes))\n    removed_boxes = sum(e[\'action\'] == \'remove_duplicate\' for changes in grouped.values() for e in changes)\n    result = dict(schema_version=2, kind=\'car35_reviewed_label_patch\', source_manifest_sha256=expected_manifest_sha256,\n                  names=list(CAR35), changed_images=len(changed), changed_boxes=len(edits), changes=changed,\n                  removed_boxes=removed_boxes, relabeled_boxes=len(edits)-removed_boxes,\n                  standalone_training_dataset=False, independent_field_test=False,\n                  source_modified=False, split_changed=False,\n                  usage=\'Review overlay only. Materialize a separately identified dataset on a writable copy before training; \'\n                        \'preserve the original V4 ZIP/directory and update manifest hashes/counts. Never apply into V4 in place.\')\n    output.parent.mkdir(parents=True, exist_ok=True)\n    with tempfile.TemporaryDirectory(prefix=\'.label-patch-\', dir=output.parent) as temporary:\n        stage = Path(temporary)/\'patch\'; stage.mkdir()\n        for relative, content in content_by_path.items():\n            target = stage/PurePosixPath(relative); target.parent.mkdir(parents=True, exist_ok=True)\n            target.write_bytes(content)\n        write_json(stage/\'patch.json\', result)\n        with (stage/\'corrections.csv\').open(\'w\', newline=\'\', encoding=\'utf-8-sig\') as stream:\n            fields = [\'image_sha256\', \'split\', \'action\', \'line_index\', \'retained_line_index\', \'from_class\', \'from_name\', \'to_class\', \'to_name\', \'reason\']\n            writer = csv.DictWriter(stream, fieldnames=fields); writer.writeheader()\n            for row in changed:\n                for edit in row[\'edits\']:\n                    writer.writerow(dict(image_sha256=row[\'image_sha256\'], split=row[\'split\'],\n                                         action=edit[\'action\'], retained_line_index=edit.get(\'retained_line_index\', \'\'),\n                                         line_index=edit[\'line_index\'], from_class=edit[\'from_class\'],\n                                         from_name=CAR35[edit[\'from_class\']], to_class=edit[\'to_class\'],\n                                         to_name=CAR35[edit[\'to_class\']], reason=edit[\'reason\']))\n        (stage/\'README.md\').write_text(\'# CAR35 reviewed label overlay\\n\\n\'+result[\'usage\']+\'\\n\', encoding=\'utf-8\')\n        if output.exists():\n            raise FileExistsError(output)\n        stage.rename(output)\n    return result\n', 'training/data/car35_revision.py': '"""Materialize reviewed labels from an immutable V4 ZIP into a new development dataset."""\nfrom collections import Counter\nfrom contextlib import contextmanager\nfrom copy import deepcopy\nimport csv\nimport hashlib\nimport json\nfrom pathlib import Path, PurePosixPath\nimport re\nimport shutil\nimport tempfile\nimport zipfile\n\nfrom training.data.common import CAR35, SPLITS, digest_file, write_json\nfrom training.data.car35_bundle import restore_car35_bundle, verify_car35\nfrom training.data.label_patch import create_label_patch\n\n\nREVISION_KIND = \'car35_reviewed_development_v1\'\n\n\ndef _sha(value):\n    if not isinstance(value, str) or not re.fullmatch(\'[a-f0-9]{64}\', value):\n        raise ValueError(\'Invalid revision SHA-256\')\n    return value\n\n\ndef _version(value):\n    if not isinstance(value, str) or not re.fullmatch(r\'[A-Za-z0-9][A-Za-z0-9_.-]{0,100}\', value):\n        raise ValueError(\'Supply a safe, separately identified dataset version\')\n    return value\n\n\ndef revision_manifest(source, patch, *, source_sha, patch_document_sha, patch_archive_sha, dataset_version):\n    """Deterministic identity; preserve every source image, group and split."""\n    _version(dataset_version)\n    for value in (source_sha, patch_document_sha, patch_archive_sha): _sha(value)\n    if (source.get(\'kind\') != \'car35_folder_development_v4\' or source.get(\'names\') != CAR35 or\n            source.get(\'independent_field_test\') is not False or\n            patch.get(\'kind\') != \'car35_reviewed_label_patch\' or patch.get(\'schema_version\') != 2 or\n            patch.get(\'source_manifest_sha256\') != source_sha or patch.get(\'names\') != CAR35 or\n            patch.get(\'split_changed\') is not False or patch.get(\'independent_field_test\') is not False):\n        raise ValueError(\'Invalid revision source/patch provenance\')\n    result = deepcopy(source)\n    rows = {r[\'sha256\']:r for r in result[\'images\']}\n    counts = {s:Counter({int(k):v for k,v in source[\'class_counts\'][s].items()}) for s in SPLITS}\n    seen, edited, removed, changed = set(), set(), 0, 0\n    changes = patch.get(\'changes\', [])\n    if not changes or len(changes) != patch.get(\'changed_images\'):\n        raise ValueError(\'Invalid patch image inventory\')\n    for item in changes:\n        sha = item[\'image_sha256\'];row = rows.get(sha)\n        if (row is None or sha in seen or item[\'split\'] != row[\'split\'] or item[\'label\'] != row[\'label\'] or\n                item[\'old_sha256\'] != row[\'label_sha256\']):\n            raise ValueError(\'Patch differs from source revision inventory\')\n        seen.add(sha);row[\'label_sha256\'] = _sha(item[\'new_sha256\'])\n        if not item.get(\'edits\'): raise ValueError(\'Patch has no reviewed edits\')\n        for edit in item[\'edits\']:\n            index = edit.get(\'line_index\');before,after = edit.get(\'from_class\'),edit.get(\'to_class\')\n            action = edit.get(\'action\')\n            if (edit.get(\'image_sha256\') != sha or edit.get(\'reviewed\') is not True or\n                    type(index) is not int or index < 0 or (sha,index) in edited or\n                    any(type(c) is not int or not 0 <= c < 35 for c in (before,after)) or\n                    action not in (\'replace_class\',\'remove_duplicate\') or\n                    not isinstance(edit.get(\'reason\'),str) or not edit[\'reason\'].strip()):\n                raise ValueError(\'Invalid reviewed revision edit\')\n            edited.add((sha,index));counts[row[\'split\']][before] -= 1\n            if action == \'replace_class\':\n                counts[row[\'split\']][after] += 1;changed += 1\n            else:\n                removed += 1\n    if (patch.get(\'removed_boxes\') != removed or patch.get(\'relabeled_boxes\') != changed or\n            patch.get(\'changed_boxes\') != removed+changed or\n            any(v < 0 for c in counts.values() for v in c.values())):\n        raise ValueError(\'Patch revision counts are inconsistent\')\n    result.update(kind=REVISION_KIND, dataset_version=dataset_version,\n        class_counts={s:{str(c):n for c,n in sorted(counts[s].items()) if n} for s in SPLITS},\n        missing_classes={s:[name for c,name in enumerate(CAR35) if not counts[s][c]] for s in SPLITS},\n        label_revision=dict(schema_version=1, source_manifest_sha256=source_sha,\n            patch_document_sha256=patch_document_sha, patch_archive_sha256=patch_archive_sha,\n            changed_images=len(changes), relabeled_boxes=changed, removed_boxes=removed,\n            split_changed=False, validation_labels_changed=any(c[\'split\']==\'val\' for c in changes),\n            test_labels_changed=any(c[\'split\']==\'test\' for c in changes)),\n        limitations=source.get(\'limitations\',\'Development split; scene overlap may remain.\')+\n            \' Reviewed label revision; image/group/split membership is unchanged. \'\n            \'Changed validation/test labels change the evaluation reference. Not an independent field test.\')\n    return result\n\n\ndef validate_revision_provenance(root, manifest):\n    root = Path(root)\n    original, patch = root/\'provenance/v4_manifest.json\', root/\'provenance/label_patch.json\'\n    if any(p.is_symlink() or not p.resolve().is_relative_to(root.resolve()) for p in (original,patch)):\n        raise ValueError(\'Unsafe revision provenance path\')\n    revision = manifest[\'label_revision\']\n    if (digest_file(original) != revision[\'source_manifest_sha256\'] or\n            digest_file(patch) != revision[\'patch_document_sha256\']):\n        raise ValueError(\'Revision provenance hash changed\')\n    expected = revision_manifest(json.loads(original.read_text(encoding=\'utf-8\')),\n        json.loads(patch.read_text(encoding=\'utf-8\')), source_sha=revision[\'source_manifest_sha256\'],\n        patch_document_sha=revision[\'patch_document_sha256\'], patch_archive_sha=revision[\'patch_archive_sha256\'],\n        dataset_version=manifest[\'dataset_version\'])\n    if expected != manifest:\n        raise ValueError(\'Revision manifest differs from source/patch provenance\')\n\n\n@contextmanager\ndef _patch_directory(archive, expected_sha256, parent=None):\n    archive = Path(archive).resolve()\n    if digest_file(archive) != _sha(expected_sha256): raise ValueError(\'Patch ZIP checksum mismatch\')\n    with zipfile.ZipFile(archive) as source:\n        entries = source.infolist();seen = set()\n        if len(entries) > 10000 or sum(m.file_size for m in entries) > 64*1024**2:\n            raise ValueError(\'Patch ZIP exceeds labels-only size limit\')\n        for member in entries:\n            name = member.filename;relative = PurePosixPath(name)\n            if (not name or relative.is_absolute() or \'..\' in relative.parts or \'\\\\\' in name or \':\' in name or\n                    name.casefold() in seen or member.flag_bits&1 or\n                    (member.external_attr>>16)&0o170000 == 0o120000):\n                raise ValueError(\'Unsafe or duplicate patch ZIP path\')\n            seen.add(name.casefold())\n        if \'patch.json\' not in seen or source.getinfo(\'patch.json\').file_size > 1024**2:\n            raise ValueError(\'Missing or oversized patch.json\')\n        with tempfile.TemporaryDirectory(prefix=\'.car35-patch-\',dir=parent) as temporary:\n            root = Path(temporary)\n            for member in entries:\n                if member.is_dir(): continue\n                target = root/member.filename;target.parent.mkdir(parents=True,exist_ok=True)\n                with source.open(member) as stream,target.open(\'wb\') as dest: shutil.copyfileobj(stream,dest)\n            yield root\n\n\ndef _prepare_revision(root, patch_root, *, source_sha, patch_sha, dataset_version):\n    raw = (root/\'manifest.json\').read_bytes()\n    if hashlib.sha256(raw).hexdigest() != _sha(source_sha): raise ValueError(\'Source manifest SHA-256 mismatch\')\n    patch_raw = (patch_root/\'patch.json\').read_bytes();patch = json.loads(patch_raw)\n    with tempfile.TemporaryDirectory(prefix=\'.replay-patch-\',dir=patch_root.parent) as temporary:\n        replay_root = Path(temporary)/\'replayed\'\n        edits = [edit for item in patch[\'changes\'] for edit in item[\'edits\']]\n        replay = create_label_patch(root,replay_root,edits,expected_manifest_sha256=source_sha)\n        if replay != patch: raise ValueError(\'Patch metadata differs from reviewed edit replay\')\n        expected_labels = {item[\'label\'] for item in replay[\'changes\']}\n        if {p.relative_to(patch_root).as_posix() for p in (patch_root/\'labels\').rglob(\'*\') if p.is_file()} != expected_labels:\n            raise ValueError(\'Patch label inventory differs from reviewed edits\')\n        for name in expected_labels:\n            if (replay_root/name).read_bytes() != (patch_root/name).read_bytes():\n                raise ValueError(\'Patch label bytes differ from reviewed edit replay\')\n    manifest = revision_manifest(json.loads(raw),patch,source_sha=source_sha,\n        patch_document_sha=hashlib.sha256(patch_raw).hexdigest(), patch_archive_sha=patch_sha,\n        dataset_version=dataset_version)\n    manifest_bytes = json.dumps(manifest,ensure_ascii=False,indent=2,allow_nan=False).encode(\'utf-8\')\n    return manifest,manifest_bytes,raw,patch_raw\n\n\ndef preview_car35_revision(root, patch_archive, *, expected_source_manifest_sha256,\n                           expected_patch_sha256, dataset_version):\n    """Verify source/patch read-only and predict the exact new manifest checksum."""\n    root = Path(root).resolve();_version(dataset_version)\n    verify_car35(root,expected_manifest_sha256=expected_source_manifest_sha256)\n    with _patch_directory(patch_archive,expected_patch_sha256) as patch_root:\n        manifest,raw,_,_ = _prepare_revision(root,patch_root,source_sha=expected_source_manifest_sha256,\n            patch_sha=expected_patch_sha256,dataset_version=dataset_version)\n    return dict(manifest=manifest,manifest_sha256=hashlib.sha256(raw).hexdigest(),\n        images=len(manifest[\'images\']),boxes=sum(sum(c.values()) for c in manifest[\'class_counts\'].values()),\n        dataset_version=dataset_version,materialized=False)\n\n\ndef restore_car35_revision(archive, patch_archive, output, *, expected_sha256,\n        expected_source_manifest_sha256, expected_patch_sha256, dataset_version, expected_manifest_sha256=\'\'):\n    """Restore once into owned staging, replay reviewed edits, verify, then publish."""\n    output = Path(output).resolve();_version(dataset_version)\n    if output.exists(): raise FileExistsError(output)\n    output.parent.mkdir(parents=True,exist_ok=True)\n    with _patch_directory(patch_archive,expected_patch_sha256,output.parent) as patch_root:\n        with tempfile.TemporaryDirectory(prefix=\'.car35-revision-\',dir=output.parent) as temporary:\n            stage = Path(temporary)/\'dataset\'\n            restore_car35_bundle(archive,stage,expected_sha256=expected_sha256,\n                                 expected_manifest_sha256=expected_source_manifest_sha256)\n            manifest,raw,original,patch_raw = _prepare_revision(stage,patch_root,\n                source_sha=expected_source_manifest_sha256,patch_sha=expected_patch_sha256,dataset_version=dataset_version)\n            if expected_manifest_sha256 and hashlib.sha256(raw).hexdigest()!=expected_manifest_sha256:\n                raise ValueError(\'Revised manifest SHA-256 mismatch\')\n            provenance = stage/\'provenance\';provenance.mkdir()\n            (provenance/\'v4_manifest.json\').write_bytes(original)\n            (provenance/\'label_patch.json\').write_bytes(patch_raw)\n            # Keep historical previews/reports explicitly separate from revised labels.\n            historical = provenance/\'v4_reports\';historical.mkdir()\n            for name in (\'README.md\',\'report.html\',\'dataset_report.json\',\'previews\'):\n                if (stage/name).exists(): (stage/name).rename(historical/name)\n            for item in json.loads(patch_raw)[\'changes\']:\n                shutil.copyfile(patch_root/item[\'label\'],stage/item[\'label\'])\n            for split in SPLITS:\n                for suffix in (\'.cache\',\'.cache.npy\'): (stage/\'labels\'/(split+suffix)).unlink(missing_ok=True)\n            (stage/\'manifest.json\').write_bytes(raw)\n            with (stage/\'class_counts.csv\').open(\'w\',newline=\'\',encoding=\'utf-8-sig\') as stream:\n                writer=csv.DictWriter(stream,fieldnames=[\'class_id\',\'name\',*SPLITS]);writer.writeheader()\n                for i,name in enumerate(CAR35):\n                    writer.writerow(dict(class_id=i,name=name,**{s:manifest[\'class_counts\'][s].get(str(i),0) for s in SPLITS}))\n            (stage/\'README.md\').write_text(\'# \'+dataset_version+\'\\n\\nReviewed label revision of CAR35 V4. \'\n                \'Images/groups/splits are unchanged; historical reports are under provenance/v4_reports. \'\n                \'Changed holdout labels change the evaluation reference. Not an independent field test.\\n\',encoding=\'utf-8\')\n            info=verify_car35(stage,expected_manifest_sha256=hashlib.sha256(raw).hexdigest())\n            write_json(stage/\'dataset_report.json\',info)\n            if output.exists(): raise FileExistsError(output)\n            stage.rename(output)\n    return info\n\n\ndef pack_car35_revision(root, target):\n    root,target = Path(root).resolve(),Path(target).resolve()\n    if target.is_relative_to(root) or root.is_relative_to(target): raise ValueError(\'ZIP must be outside dataset\')\n    sidecar=target.with_suffix(target.suffix+\'.sha256\')\n    if target.exists() or sidecar.exists(): raise FileExistsError(target)\n    info=verify_car35(root)\n    if info[\'kind\'] != REVISION_KIND: raise ValueError(\'Expected a reviewed CAR35 revision\')\n    caches={root/\'labels\'/(s+suffix) for s in SPLITS for suffix in (\'.cache\',\'.cache.npy\')}\n    paths=sorted(p for p in root.rglob(\'*\') if p.is_file() and p not in caches)\n    if any(p.is_symlink() or not p.resolve().is_relative_to(root) for p in paths): raise ValueError(\'Unsafe revision file\')\n    target.parent.mkdir(parents=True,exist_ok=True)\n    if shutil.disk_usage(target.parent).free < sum(p.stat().st_size for p in paths)+1024**3:\n        raise ValueError(\'Not enough disk space to pack CAR35 revision\')\n    with tempfile.TemporaryDirectory(prefix=\'.pack-revision-\',dir=target.parent) as temporary:\n        archive=Path(temporary)/\'dataset.zip\'\n        with zipfile.ZipFile(archive,\'x\',compression=zipfile.ZIP_STORED,allowZip64=True) as dest:\n            for path in paths: dest.write(path,path.relative_to(root).as_posix())\n        checksum=digest_file(archive)\n        if target.exists(): raise FileExistsError(target)\n        archive.rename(target)\n    sidecar.write_text(checksum+\'\\n\',encoding=\'utf-8\')\n    return dict(info,zip_sha256=checksum)\n', 'training/experiments/car35.py': '"""Train/export prepared CAR35 development data; never changes application serving."""\nimport json\nimport math\nfrom pathlib import Path\nimport zipfile\n\nfrom training.data.car35_bundle import verify_car35\nfrom training.data.common import digest_file, read_json, write_json\n\n\ndef train_car35(root,run_dir,*,epochs=80,imgsz=640,batch=8,device=\'0\',resume=False,yolo_factory=None):\n    root,run_dir=Path(root).resolve(),Path(run_dir).resolve()\n    if run_dir.is_relative_to(root) or root.is_relative_to(run_dir):raise ValueError(\'Run must be outside dataset\')\n    if epochs<1 or imgsz<32 or batch<1:raise ValueError(\'Invalid training settings\')\n    dataset=verify_car35(root)\n    settings=dict(model=\'yolo11s.pt\',epochs=epochs,imgsz=imgsz,batch=batch,seed=42)\n    request=run_dir/\'training_request.json\'\n    if resume:\n        if not request.is_file() or not (run_dir/\'weights/last.pt\').is_file():\n            raise ValueError(\'Resume requires last.pt and training_request.json in this runtime\')\n        previous=read_json(request)\n        if previous[\'dataset\'][\'manifest_sha256\']!=dataset[\'manifest_sha256\']:raise ValueError(\'Resume dataset changed\')\n        if previous[\'settings\']!=settings:raise ValueError(\'Resume settings changed\')\n        weights=run_dir/\'weights/last.pt\'\n    else:\n        if run_dir.exists():raise FileExistsError(run_dir)\n        weights=\'yolo11s.pt\'\n    if yolo_factory is None:\n        from ultralytics import YOLO\n        yolo_factory=YOLO\n    detector=yolo_factory(str(weights))\n    if resume:\n        names=detector.names;names=[names[i] for i in range(len(names))] if isinstance(names,dict) else list(names)\n        if detector.task!=\'detect\' or names!=dataset[\'names\']:raise ValueError(\'Resume checkpoint taxonomy differs\')\n    if not resume:\n        run_dir.mkdir(parents=True)\n        write_json(request,dict(dataset=dataset,settings=settings,test_evaluated=False))\n    options=dict(data=str(root/\'data.yaml\'),epochs=epochs,imgsz=imgsz,batch=batch,device=device,\n        project=str(run_dir.parent),name=run_dir.name,exist_ok=True,split=\'val\',workers=2,\n        seed=42,deterministic=True,cache=False,amp=True,patience=20,save=True,save_period=5,\n        fliplr=0.,flipud=0.,degrees=0.,shear=0.,perspective=0.,mixup=0.,mosaic=.5,\n        close_mosaic=min(10,epochs),plots=True)\n    if resume:options[\'resume\']=True\n    detector.train(**options)\n    best=run_dir/\'weights/best.pt\'\n    if not best.is_file():raise ValueError(\'Training did not produce best.pt\')\n    return best\n\n\ndef evaluate_car35(checkpoint,root,output,*,split=\'val\',allow_test=False,imgsz=640,batch=8,device=\'0\',yolo_factory=None):\n    if split not in (\'val\',\'test\') or split==\'test\' and not allow_test:raise ValueError(\'Held-out test requires explicit opt-in\')\n    root,output,checkpoint=Path(root).resolve(),Path(output).resolve(),Path(checkpoint).resolve()\n    if output.exists():raise FileExistsError(output)\n    if output.is_relative_to(root) or root.is_relative_to(output):raise ValueError(\'Evaluation must be outside dataset\')\n    dataset=verify_car35(root)\n    if yolo_factory is None:\n        from ultralytics import YOLO\n        yolo_factory=YOLO\n    detector=yolo_factory(str(checkpoint));names=detector.names\n    names=[names[i] for i in range(len(names))] if isinstance(names,dict) else list(names)\n    if detector.task!=\'detect\' or names!=dataset[\'names\']:raise ValueError(\'Evaluation checkpoint taxonomy differs\')\n    output.mkdir(parents=True)\n    result=detector.val(data=str(root/\'data.yaml\'),split=split,imgsz=imgsz,batch=batch,device=device,\n                        project=str(output.parent),name=output.name,exist_ok=True,plots=True)\n    report=dict(dataset=dataset,split=split,split_images=dataset[\'split_images\'][split],\n                checkpoint_sha256=digest_file(checkpoint),independent_field_test=False,\n                metrics={k:float(v) if math.isfinite(float(v)) else None for k,v in result.results_dict.items()})\n    box=getattr(result,\'box\',None)\n    if box is not None:\n        report[\'per_class\']=[dict(class_id=int(c),name=names[int(c)],precision=float(box.p[j]),\n            recall=float(box.r[j]),map50=float(box.ap50[j]),map50_95=float(box.ap[j])) for j,c in enumerate(box.ap_class_index)]\n    write_json(output/\'evaluation_report.json\',report)\n    return report\n\n\ndef export_car35_run(run_dir,root,target):\n    run_dir,root,target=Path(run_dir).resolve(),Path(root).resolve(),Path(target).resolve()\n    if target.exists():raise FileExistsError(target)\n    if target.is_relative_to(run_dir) or target.is_relative_to(root):raise ValueError(\'Export must be outside run/dataset\')\n    if not any((run_dir/\'weights\'/n).is_file() for n in (\'best.pt\',\'last.pt\')):raise ValueError(\'No checkpoint to export yet\')\n    dataset=verify_car35(root)\n    request=read_json(run_dir/\'training_request.json\')\n    if request[\'dataset\'][\'manifest_sha256\']!=dataset[\'manifest_sha256\']:raise ValueError(\'Export dataset changed\')\n    handoff=dict(schema_version=1,dataset=dataset,settings=request[\'settings\'],independent_field_test=False,\n        checkpoints={n:digest_file(run_dir/\'weights\'/n) for n in (\'best.pt\',\'last.pt\') if (run_dir/\'weights\'/n).is_file()},\n        limitations=dataset[\'limitations\'])\n    target.parent.mkdir(parents=True,exist_ok=True)\n    with zipfile.ZipFile(target,\'x\',compression=zipfile.ZIP_DEFLATED,allowZip64=True) as archive:\n        for path in sorted(run_dir.rglob(\'*\')):\n            if path.is_file() and not path.is_symlink():archive.write(path,\'run/\'+path.relative_to(run_dir).as_posix())\n        for name in (\'manifest.json\',\'data.yaml\',\'dataset_report.json\',\'class_counts.csv\',\'README.md\'):\n            if (root/name).is_file():archive.write(root/name,\'dataset/\'+name)\n        for name in (\'v4_manifest.json\',\'label_patch.json\'):\n            if (root/\'provenance\'/name).is_file():archive.write(root/\'provenance\'/name,\'dataset/provenance/\'+name)\n        archive.writestr(\'handoff.json\',json.dumps(handoff,ensure_ascii=False,indent=2))\n    return target\n'}
PIPELINE_ID = hashlib.sha256(json.dumps(TRAINING_FILES, sort_keys=True).encode()).hexdigest()
CODE_ROOT = WORK / 'pipeline' / PIPELINE_ID
for relative, source in TRAINING_FILES.items():
    target = CODE_ROOT / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source, encoding='utf-8')
for name in list(sys.modules):
    if name in ('training', 'training.data', 'training.experiments') or name in (
        'training.data.common', 'training.data.downloads', 'training.data.car35_bundle',
        'training.data.label_patch', 'training.data.car35_revision', 'training.experiments.car35'):
        del sys.modules[name]
sys.path.insert(0, str(CODE_ROOT))
importlib.invalidate_caches()
from training.data.downloads import download_drive_zip, drive_file_id
from training.data.car35_bundle import restore_car35_bundle, verify_car35, inspect_car35_archive
from training.data.car35_revision import restore_car35_revision, pack_car35_revision
from training.experiments.car35 import train_car35, evaluate_car35, export_car35_run
print('Pipeline:', PIPELINE_ID[:16])


In [ ]:
expected_manifest = EXPECTED_MANIFEST_SHA256
DATASET_ROOT = WORK / 'dataset'
if APPLY_REVIEWED_PATCH:
    assert PATCH_SHA256 and EXPECTED_REVISED_MANIFEST_SHA256, 'Cần checksum patch và manifest revision đã rà.'
    assert REVISED_DATASET_VERSION and Path(REVISED_DATASET_VERSION).name == REVISED_DATASET_VERSION and REVISED_DATASET_VERSION not in ('.','..'), 'Dataset version không hợp lệ.'
    DATASET_ROOT = WORK / REVISED_DATASET_VERSION
    expected_manifest = EXPECTED_REVISED_MANIFEST_SHA256
if DATASET_ROOT.exists():
    dataset_info = verify_car35(DATASET_ROOT, expected_manifest_sha256=expected_manifest)
else:
    assert SOURCE_URL.strip(), 'Điền SOURCE_URL bằng link file ZIP CAR35 trên Drive.'
    if APPLY_REVIEWED_PATCH:
        assert PATCH_URL.strip(), 'Điền PATCH_URL bằng link ZIP patch R3 trước khi tải dataset.'
    if APPLY_REVIEWED_PATCH:
        patch_archive = download_drive_zip(PATCH_URL, WORK / 'downloads' / ('patch_' + drive_file_id(PATCH_URL) + '.zip'),
            expected_sha256=PATCH_SHA256)
    # Container SHA may differ after repacking. Never trust a downloaded
    # manifest as a new reference: its independently pinned hash must match.
    archive = download_drive_zip(SOURCE_URL, WORK / 'downloads' / ('car35_' + drive_file_id(SOURCE_URL) + '.zip'))
    source_identity = inspect_car35_archive(archive, expected_sha256=DATASET_SHA256,
        expected_manifest_sha256=EXPECTED_MANIFEST_SHA256, allow_repacked=ALLOW_REPACKED_SOURCE)
    (WORK / 'source_archive_verification.json').write_text(json.dumps(source_identity, indent=2), encoding='utf-8')
    print('Source ZIP SHA-256:', source_identity['archive_sha256'])
    if source_identity['repacked']:
        print('ZIP khác cách đóng gói; manifest khớp. Tiếp tục kiểm tra toàn bộ ảnh/nhãn trước khi dùng.')
    if APPLY_REVIEWED_PATCH:
        print('Khôi phục V4, replay patch và kiểm tra revision; không tạo bản sao thứ hai của toàn bộ ảnh...', flush=True)
        dataset_info = restore_car35_revision(archive, patch_archive, DATASET_ROOT,
            expected_sha256=source_identity['archive_sha256'], expected_source_manifest_sha256=EXPECTED_MANIFEST_SHA256,
            expected_patch_sha256=PATCH_SHA256, dataset_version=REVISED_DATASET_VERSION,
            expected_manifest_sha256=EXPECTED_REVISED_MANIFEST_SHA256)
    else:
        dataset_info = restore_car35_bundle(archive, DATASET_ROOT,
            expected_sha256=source_identity['archive_sha256'], expected_manifest_sha256=EXPECTED_MANIFEST_SHA256)
manifest = json.loads((DATASET_ROOT / 'manifest.json').read_text(encoding='utf-8'))
if APPLY_REVIEWED_PATCH:
    assert manifest.get('dataset_version') == REVISED_DATASET_VERSION
    assert manifest['label_revision']['patch_archive_sha256'] == PATCH_SHA256
print('Dataset version:', dataset_info['dataset_version'], '| Manifest:', dataset_info['manifest_sha256'])
print(f"CAR35: {dataset_info['images']:,} images | {dataset_info['boxes']:,} boxes | 35 classes")
for split in ('train', 'val', 'test'):
    print(split, ':', dataset_info['split_images'][split], 'images')
print('Giữ nguyên split. Test thực địa độc lập:', dataset_info['independent_field_test'])


## 2. Train YOLO11s

Bắt đầu từ pretrained `yolo11s.pt`, vì model SupSign cũ có thể đã thấy các ảnh nay thuộc holdout V4.
Chọn GPU runtime trước khi chạy. Nếu hết VRAM, giảm BATCH ở lần chạy mới; resume phải giữ đúng settings cũ.
Checkpoint best/last và mỗi 5 epoch chỉ lưu local. Khi dừng giữa chừng, chạy cell Xuất kết quả nếu đã có weights.
RESUME chỉ dùng khi run còn trong runtime hoặc đã tự tải/khôi phục run đầy đủ; notebook không tự phục hồi checkpoint từ tài khoản khác.


In [ ]:
if DEVICE != 'cpu':
    import torch
    assert torch.cuda.is_available(), 'Chọn Runtime → Change runtime type → GPU trước khi train.'
try:
    best_checkpoint = train_car35(DATASET_ROOT, RUN_DIR, epochs=EPOCHS, imgsz=IMAGE_SIZE,
        batch=BATCH, device=DEVICE, resume=RESUME)
    print('Best:', best_checkpoint)
except KeyboardInterrupt:
    print('Đã dừng. Chạy cell Xuất kết quả để giữ checkpoint nếu run đã tạo weights.')


## 3. Validation và test riêng

Validation được dùng để xem lỗi/chọn cấu hình. Test mặc định tắt; chỉ bật khi đã chốt.
Đủ 35 tên lớp không có nghĩa đủ mẫu kiểm thử. Giữ báo cáo giới hạn scene overlap và lớp hiếm khi đọc metric.


In [ ]:
from datetime import datetime, timezone
def fresh_evaluation_directory(label):
    output = RUN_DIR / label
    if output.exists():
        output = RUN_DIR / (label + '_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
    return output
checkpoint = RUN_DIR / 'weights' / 'best.pt'
if checkpoint.is_file():
    val_report = evaluate_car35(checkpoint, DATASET_ROOT, fresh_evaluation_directory('validation'),
        imgsz=IMAGE_SIZE, batch=BATCH, device=DEVICE)
    print('Validation:', val_report['metrics'])
    if RUN_HELDOUT_TEST:
        test_report = evaluate_car35(checkpoint, DATASET_ROOT, fresh_evaluation_directory('test'), split='test', allow_test=True,
            imgsz=IMAGE_SIZE, batch=BATCH, device=DEVICE)
        print('Development test:', test_report['metrics'])
else:
    print('Chưa có best.pt để evaluate; có thể xuất last.pt nếu đã train một phần.')


## 4. Xuất kết quả về máy

ZIP gồm best.pt/last.pt, checkpoint định kỳ, plots/results.csv, training_request, evaluation reports và manifest/checksum.
Tải ZIP trước khi đóng runtime. Sau khi bạn gửi kết quả, mới rà chất lượng và mapping để tích hợp; notebook không đổi model đang phục vụ.


In [ ]:
from datetime import datetime, timezone
name = RUN_NAME + '_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
if RUN_DIR.is_dir() and (WORK / 'source_archive_verification.json').is_file():
    shutil.copy2(WORK / 'source_archive_verification.json', RUN_DIR / 'source_archive_verification.json')
RESULT_ZIP = export_car35_run(RUN_DIR, DATASET_ROOT, WORK / (name + '.zip'))
print('Training result:', RESULT_ZIP)
if DOWNLOAD_RESULTS:
    from google.colab import files
    files.download(str(RESULT_ZIP))
